# Notebook 12: Neural Collaborative Filtering (NCF) - CORREGIDO

Este notebook entrena variantes de NCF con diferentes tamaños de embedding.

**METODOLOGÍA CORRECTA:** train_test_split INTERNO (como Notebook 7)

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from codecarbon import EmissionsTracker

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

TensorFlow version: 2.20.0
GPU disponible: []


## Configuración

In [5]:
DATA_PATH = Path("../data/processed")
SUBSAMPLED_PATH = DATA_PATH / "subsampled_informed"
RESULTS_PATH = Path("../results")
RESULTS_PATH.mkdir(exist_ok=True, parents=True)

EMBEDDING_SIZES = [20, 50, 100]
FRACTIONS = [10, 25, 50, 75]
EPOCHS = 10
BATCH_SIZE = 256
LEARNING_RATE = 0.001

## Modelo NCF

In [6]:
def build_ncf_model(n_users, n_items, embedding_size=50):
    """
    Construye modelo NCF (Neural Collaborative Filtering)
    
    Arquitectura:
    - Embedding de usuarios y items
    - Concatenación
    - Capas densas con dropout
    - Salida: rating predicho [1-5]
    """
    user_input = layers.Input(shape=(1,), name='user_input')
    item_input = layers.Input(shape=(1,), name='item_input')
    
    user_embedding = layers.Embedding(
        input_dim=n_users + 1,
        output_dim=embedding_size,
        name='user_embedding'
    )(user_input)
    user_vec = layers.Flatten(name='user_flatten')(user_embedding)
    
    item_embedding = layers.Embedding(
        input_dim=n_items + 1,
        output_dim=embedding_size,
        name='item_embedding'
    )(item_input)
    item_vec = layers.Flatten(name='item_flatten')(item_embedding)
    
    concat = layers.Concatenate(name='concat')([user_vec, item_vec])
    
    dense1 = layers.Dense(128, activation='relu', name='dense1')(concat)
    dropout1 = layers.Dropout(0.2, name='dropout1')(dense1)
    
    dense2 = layers.Dense(64, activation='relu', name='dense2')(dropout1)
    dropout2 = layers.Dropout(0.2, name='dropout2')(dense2)
    
    dense3 = layers.Dense(32, activation='relu', name='dense3')(dropout2)
    
    output = layers.Dense(1, activation='linear', name='output')(dense3)
    
    model = keras.Model(
        inputs=[user_input, item_input],
        outputs=output,
        name=f'NCF_emb{embedding_size}'
    )
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=['mae']
    )
    
    return model

## Funciones de evaluación

In [7]:
def precision_recall_at_k(predictions_dict, test_df, k=10, threshold=3.5):
    """
    Calcula Precision@K y Recall@K
    
    predictions_dict: {user_id: [(item_id, predicted_rating), ...]}
    test_df: DataFrame con columnas [userId, movieId, rating]
    """
    precisions = []
    recalls = []
    
    for user_id, pred_list in predictions_dict.items():
        user_test = test_df[test_df['userId'] == user_id]
        true_items = set(user_test[user_test['rating'] >= threshold]['movieId'].values)
        
        if len(true_items) == 0:
            continue
        
        pred_list_sorted = sorted(pred_list, key=lambda x: x[1], reverse=True)
        top_k_items = set([item_id for item_id, _ in pred_list_sorted[:k]])
        
        hits = top_k_items & true_items
        precision = len(hits) / k if k > 0 else 0
        recall = len(hits) / len(true_items) if len(true_items) > 0 else 0
        
        precisions.append(precision)
        recalls.append(recall)
    
    return np.mean(precisions) if precisions else 0, np.mean(recalls) if recalls else 0

## Función principal de entrenamiento

In [8]:
def train_and_evaluate_ncf(file_path, embedding_size, fraction):
    """
    Entrena y evalúa NCF usando METODOLOGÍA CORRECTA:
    - Carga subset completo
    - Split interno (80/20)
    - Evalúa sobre testset del MISMO subset
    """
    print(f"\n{'='*80}")
    print(f" NCF emb={embedding_size} - Fracción {fraction}%")
    print(f"{'='*80}")
    
    df = pd.read_csv(file_path)
    print(f"   Cargado: {len(df):,} ratings")
    print(f"   Usuarios: {df['userId'].nunique():,}")
    print(f"   Películas: {df['movieId'].nunique():,}")
    
    from sklearn.model_selection import train_test_split
    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
    print(f"  Split: Train={len(train_df):,}, Test={len(test_df):,}")
    
    user_ids = df['userId'].unique()
    item_ids = df['movieId'].unique()
    
    user_map = {uid: idx for idx, uid in enumerate(user_ids)}
    item_map = {iid: idx for idx, iid in enumerate(item_ids)}
    
    train_df['user_idx'] = train_df['userId'].map(user_map)
    train_df['item_idx'] = train_df['movieId'].map(item_map)
    test_df['user_idx'] = test_df['userId'].map(user_map)
    test_df['item_idx'] = test_df['movieId'].map(item_map)
    
    n_users = len(user_ids)
    n_items = len(item_ids)
    
    X_train_user = train_df['user_idx'].values
    X_train_item = train_df['item_idx'].values
    y_train = train_df['rating'].values
    
    X_test_user = test_df['user_idx'].values
    X_test_item = test_df['item_idx'].values
    y_test = test_df['rating'].values
    
    print(f" Construyendo modelo NCF (emb={embedding_size})...")
    model = build_ncf_model(n_users, n_items, embedding_size)
    
    print(f" Entrenando modelo...")
    tracker = EmissionsTracker(
        project_name=f"ncf_emb{embedding_size}_f{fraction}",
        log_level='error',
        save_to_file=False
    )
    tracker.start()
    
    history = model.fit(
        [X_train_user, X_train_item],
        y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=0,
        validation_split=0.1
    )
    
    emissions = tracker.stop()
    
    print(f" Evaluando modelo...")
    y_pred = model.predict([X_test_user, X_test_item], verbose=0).flatten()
    
    y_pred = np.clip(y_pred, 1.0, 5.0)
    
    rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))
    mae = np.mean(np.abs(y_test - y_pred))
    
    print(f" Calculando métricas de ranking...")
    
    predictions_dict = defaultdict(list)
    
    for user_id in test_df['userId'].unique():
        user_idx = user_map[user_id]
        
        all_item_indices = np.arange(n_items)
        user_array = np.full(n_items, user_idx)
        
        item_preds = model.predict([user_array, all_item_indices], verbose=0).flatten()
        item_preds = np.clip(item_preds, 1.0, 5.0)
        
        item_map_inv = {idx: iid for iid, idx in item_map.items()}
        for item_idx, pred in enumerate(item_preds):
            item_id = item_map_inv[item_idx]
            predictions_dict[user_id].append((item_id, pred))
    
    precision, recall = precision_recall_at_k(predictions_dict, test_df, k=10, threshold=3.5)
    
    result = {
        'method': f'NCF_emb{embedding_size}',
        'embedding_size': embedding_size,
        'fraction': fraction,
        'rmse': rmse,
        'mae': mae,
        'precision_at_10': precision,
        'recall_at_10': recall,
        'co2_kg': emissions if emissions else 0.0,
        'n_train': len(train_df),
        'n_test': len(test_df),
        'n_users': n_users,
        'n_items': n_items
    }
    
    print(f"   Resultados:")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE: {mae:.4f}")
    print(f"   Precision@10: {precision:.4f}")
    print(f"   Recall@10: {recall:.4f}")
    print(f"   CO2: {emissions:.6f} kg" if emissions else "   CO2: No disponible")
    
    del model
    tf.keras.backend.clear_session()
    
    return result

## Ejecutar experimentos NCF

In [9]:
print("\n" + "="*80)
print("EVALUANDO VARIANTES NCF")
print("="*80)

ncf_results = []

for embedding_size in EMBEDDING_SIZES:
    for fraction in FRACTIONS:
        file_path = SUBSAMPLED_PATH / f"ratings_top_users_{fraction}.csv"
        
        if not file_path.exists():
            print(f"  Archivo no encontrado: {file_path}")
            continue
        
        result = train_and_evaluate_ncf(file_path, embedding_size, fraction)
        ncf_results.append(result)

df_ncf = pd.DataFrame(ncf_results)
output_file = RESULTS_PATH / "ncf_metrics.csv"
df_ncf.to_csv(output_file, index=False)

print(f"\n Resultados NCF guardados en: {output_file}")
print(f"\n Resumen NCF:")
print(df_ncf.to_string(index=False))


EVALUANDO VARIANTES NCF

 NCF emb=20 - Fracción 10%
   Cargado: 381,007 ratings
   Usuarios: 604
   Películas: 3,624
  Split: Train=304,805, Test=76,202
 Construyendo modelo NCF (emb=20)...


[codecarbon WARNING @ 23:58:19] Multiple instances of codecarbon are allowed to run at the same time.


 Entrenando modelo...
 Evaluando modelo...
 Calculando métricas de ranking...
   Resultados:
   RMSE: 0.8694
   MAE: 0.6795
   Precision@10: 0.0975
   Recall@10: 0.0169
   CO2: 0.000257 kg


 NCF emb=20 - Fracción 25%
   Cargado: 640,906 ratings
   Usuarios: 1,510
   Películas: 3,656
  Split: Train=512,724, Test=128,182
 Construyendo modelo NCF (emb=20)...
 Entrenando modelo...
 Evaluando modelo...
 Calculando métricas de ranking...
   Resultados:
   RMSE: 0.8660
   MAE: 0.6815
   Precision@10: 0.0664
   Recall@10: 0.0170
   CO2: 0.000419 kg

 NCF emb=20 - Fracción 50%
   Cargado: 854,612 ratings
   Usuarios: 3,020
   Películas: 3,671
  Split: Train=683,689, Test=170,923
 Construyendo modelo NCF (emb=20)...
 Entrenando modelo...
 Evaluando modelo...
 Calculando métricas de ranking...
   Resultados:
   RMSE: 0.8721
   MAE: 0.6846
   Precision@10: 0.0413
   Recall@10: 0.0159
   CO2: 0.000623 kg

 NCF emb=20 - Fracción 75%
   Cargado: 954,369 ratings
   Usuarios: 4,530
   Películas: 3,691

## Verificación de resultados

In [10]:
print("\n" + "="*80)
print(" VERIFICACIÓN DE COHERENCIA")
print("="*80)

p_mean = df_ncf['precision_at_10'].mean()
r_mean = df_ncf['recall_at_10'].mean()

print(f"\n Estadísticas globales:")
print(f"   Precision@10 media: {p_mean:.4f}")
print(f"   Recall@10 media: {r_mean:.4f}")

if p_mean < 0.05:
    print(f"\n  ADVERTENCIA: Precision@10 muy baja ({p_mean:.4f})")
    print(f"   Esto podría indicar problemas en el entrenamiento o datos")
elif p_mean > 0.1:
    print(f"\n Precision@10 razonable ({p_mean:.4f})")

print("\n NOTEBOOK 12 (NCF) COMPLETADO EXITOSAMENTE")


 VERIFICACIÓN DE COHERENCIA

 Estadísticas globales:
   Precision@10 media: 0.0584
   Recall@10 media: 0.0173

 NOTEBOOK 12 (NCF) COMPLETADO EXITOSAMENTE
